In [ ]:
%pip install pandas kagglehub

### Задание 1 (синтаксис)

Для каждого покупателя получите таблицу со столбцами:

- `preferred_payment` — самый частый метод оплаты (при ничьей зафиксируйте правило, например lexicographically первый);
- `total_spent` — сумма всех трат;
- `addons_spent` — сумма на дополнительные услуги и аксессуары;
- `orders_count` — число заказов.

Отсортируйте по `total_spent` по убыванию и выведите топ-10.

#### Пример работы программы

**Output:**

```text
  customer_id  preferred_payment  total_spent  addons_spent  orders_count
0        C042        Credit Card     15420.50        890.00             7
1        C018             PayPal     14810.00        450.00             5
2        C103         Debit Card     13990.25       1200.00             6
...
```

#### Пограничные случаи

- Покупатель с одним заказом — `orders_count = 1`, `preferred_payment` — единственный метод.
- Два метода оплаты с одинаковой частотой — выберите один по зафиксированному правилу (укажите его в комментарии к коду).
- Покупатель без доп. услуг/аксессуаров — `addons_spent = 0`.
- Пропуски в сумме или методе оплаты — обработайте (отфильтруйте или заполните) и кратко опишите решение в комментарии.

### Решение

In [71]:
import kagglehub
import pandas as pd


# Download dataset
dataset = 'cameronseamons/electronic-sales-sep2023-sep2024'
data_dir = kagglehub.dataset_download(
    dataset,
    output_dir="data"
)

path = "data/Electronic_sales_Sep2023-Sep2024.csv"

print("Path to dataset files:", path)

Path to dataset files: data/Electronic_sales_Sep2023-Sep2024.csv


In [7]:
# customer_id  preferred_payment  total_spent  addons_spent  orders_count
df = pd.read_csv(path)
print(df)

       Customer ID  Age  Gender Loyalty Member Product Type      SKU  Rating  \
0             1000   53    Male             No   Smartphone  SKU1004       2   
1             1000   53    Male             No       Tablet  SKU1002       3   
2             1002   41    Male             No       Laptop  SKU1005       3   
3             1002   41    Male            Yes   Smartphone  SKU1004       2   
4             1003   75    Male            Yes   Smartphone  SKU1001       5   
...            ...  ...     ...            ...          ...      ...     ...   
19995        19996   27  Female             No   Smartphone   SMP234       4   
19996        19996   27  Female            Yes       Laptop   LTP123       4   
19997        19996   27  Female             No   Headphones   HDP456       4   
19998        19997   27    Male             No   Headphones   HDP456       1   
19999        19998   27     NaN            Yes       Laptop   LTP123       4   

      Order Status Payment Method  Tota

In [72]:
customer_spends_orders = df.groupby('Customer ID').agg(
    total_spent=('Total Price', 'sum'),
    orders_count=('Customer ID', 'size')    
).reset_index(names='customer_id')

preferred_payment = (
    df.groupby('Customer ID')['Payment Method']
# в случае нескольких методов оплаты с одинаковой частотой - выбрать первый по алфавиту
      .agg(lambda x: x.mode().sort_values().iloc[0])
      .rename_axis('customer_id')
      .reset_index(name='preferred_payment')
)

# df['Product Type'].unique() -> ['Smartphone', 'Tablet', 'Laptop', 'Smartwatch', 'Headphones']
# только наушники относятся к аксессуарам 

addons_spent = (
    df[df['Product Type'] == 'Headphones']
    .groupby('Customer ID')['Total Price'].sum()
    .reset_index(name='addons_spent')
    .rename(columns={'Customer ID': 'customer_id'})
)

result = (
    customer_spends_orders
    .merge(preferred_payment, on='customer_id', how='left')
    .merge(addons_spent, on='customer_id', how='left')
)

# Если есть пропуски - 0 или Unknown
result['addons_spent'] = result['addons_spent'].fillna(0)
result['total_spent'] = result['total_spent'].fillna(0)
result['preferred_payment'] = result['preferred_payment'].fillna('Unknown')

result = result[
    ['customer_id', 'preferred_payment', 'total_spent', 'addons_spent', 'orders_count']
].sort_values(by='total_spent', ascending=False).head(10)

result

,customer_id,preferred_payment,total_spent,addons_spent,orders_count
9728,16357,Bank Transfer,34563.70,1805.90,7
10049,16863,PayPal,33035.92,0.00,5
8060,13813,Credit Card,31830.16,0.00,5
6475,11476,PayPal,31077.61,0.00,5
7028,12276,Bank Transfer,30961.18,5056.52,6
7935,13635,Credit Card,30260.36,2167.08,5
7352,12749,Credit Card,29394.56,0.00,5
9087,15399,PayPal,29084.88,0.00,3
7054,12319,Bank Transfer,27352.32,0.00,3
12133,19996,Bank Transfer,27296.78,1805.90,6
